In [1]:
# Here we use 3 models:
#   1. 2 generative models for training
#   2. 1 bert classifier

# During  the training of 2 gen models on edgar and alice texts, 
# the bert model reads the text generated by these 2 models and classifies where it comes from

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import requests

import torch
import torch.nn as nn

from transformers import BertModel, BertTokenizer, AutoTokenizer, AutoModelForCausalLM

In [10]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [11]:
# we load the model which was saved in prev exercise
# for that we need to define that class again, create an isntance, and then load the weights
class BertForBinaryClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(BertForBinaryClassification, self).__init__()

        # Load the pretrained BERT model
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        #classification head that converts 678-d pooled output into 2 final outuputs
        self.classifier = nn.Linear(768,2)
        self.dropout = nn.Dropout(.1) #10%

        #init the weights and biases
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask = None, token_type_ids = None):
        # fwd pas through the downloaded(pretrained) BERT
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids)

        # extract the pooled output and apply dropout
        pooled_output  = self.dropout(outputs.pooler_output)

        # final push through classification layer
        logits = self.classifier(pooled_output)
        return logits

In [12]:
# create a model instance, then replace the params with those from loaded 
bert = BertForBinaryClassification().to(device)
bert.load_state_dict(torch.load('./bert_classifier_AliceVsEdgar.pt'))

# halve the memory of classifier by converting to float16 from float32
# we dont usually do this, especiialy when you first training or fine tuing a model
# but if you run to mem issues, this can be done
bert.half()

# and toggle on eval (no training ---> set to eval mode)
bert.eval()


# and need the BERT tokenizer
bertTokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# Eleuthers tokenizer
eleuTokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neo-125m')

# load in 2 GPTneos and push to GPU

modelAlice = AutoModelForCausalLM.from_pretrained('EleutherAI/gpt-neo-125m').to(device)
modelEdgar = AutoModelForCausalLM.from_pretrained('EleutherAI/gpt-neo-125m').to(device)

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
# throught the looking glass (aka alice in wonderland)
text = requests.get('https://www.gutenberg.org/cache/epub/11/pg11.txt').text
aliceTokens = eleuTokenizer.encode(text, return_tensors='pt')[0]

# edgar allan Poe
text = requests.get('https://www.gutenberg.org/cache/epub/2148/pg2148.txt').text
edgarTokens = eleuTokenizer.encode(text, return_tensors='pt')[0]

Token indices sequence length is longer than the specified maximum sequence length for this model (52951 > 2048). Running this sequence through the model will result in indexing errors


Translate text btw Eleuther to BERT

In [21]:
# ELeuther uses GPT tokenizer which is very diff from BERT tokenizer
# so we need to translate btw Bert and Eleuther tokens

# need to exclude [SEP] and [CLS] tokens that BERT iinserts by default

In [23]:
startingtext = 'Hello, my name is Raz and I like red'

# elether tokens
eleuToks = eleuTokenizer(startingtext)

# berts tokens
bertToks = bertTokenizer(startingtext)

print(f'Strating text: \n {startingtext}')
print(f'\n\n Eleuther tokens: \n {eleuToks}')
print(f"\nDecoded using Eleuther: \n {eleuTokenizer.decode(eleuToks['input_ids'])}")
print(f"\nDecoded using BERT: \n {bertTokenizer.decode(eleuToks['input_ids'])}")


print(f'\n\n BERT tokens: \n {bertToks}')
print(f"\nDecoded using BERT: \n {bertTokenizer.decode(bertToks['input_ids'])}")
print(f"\nDecoded using Eleuther: \n {eleuTokenizer.decode(bertToks['input_ids'])}")

Strating text: 
 Hello, my name is Raz and I like red


 Eleuther tokens: 
 {'input_ids': [15496, 11, 616, 1438, 318, 38058, 290, 314, 588, 2266], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Decoded using Eleuther: 
 Hello, my name is Raz and I like red

Decoded using BERT: 
 ##burgh [unused10] [unused611] ა [unused313] [unused285] [unused309] [unused583] member


 BERT tokens: 
 {'input_ids': [101, 7592, 1010, 2026, 2171, 2003, 10958, 2480, 1998, 1045, 2066, 2417, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Decoded using BERT: 
 [CLS] hello, my name is raz and i like red [SEP]

Decoded using Eleuther: 
 �lationters 50ills future spectrumiple experienceata Republicyr�


In [26]:
# text -> Eleuther toks -> text -> BERT tokens

# to Eleuther toks
startingtext = 'Hello, my name is Raz and I like red'
eleuToks = eleuTokenizer(startingtext)

# back to text
eleuReconText = eleuTokenizer.decode(eleuToks['input_ids'])

# then to bert
bertToks = bertTokenizer(eleuReconText, add_special_tokens = False) # secodn arg imp to avoid [CLS] and [SEP] tokens

# show reconstruction
bertTokenizer.decode(bertToks['input_ids'])

'hello, my name is raz and i like red'

In [27]:
# translation functions
def bert2eleu(bertToks):
    b = bertTokenizer.decode(bertToks)
    e = eleuTokenizer.encode(b,add_special_tokens=False)
    return torch.tensor(e)


def eleu2bert(eleuToks):
    e = eleuTokenizer.decode(eleuToks)
    b = bertTokenizer.encode(e,add_special_tokens=False)
    return torch.tensor(b)

#test
b2e = bert2eleu(bertToks['input_ids'])
e2b = eleu2bert(b2e)

print(eleuTokenizer.decode(b2e))
print(bertTokenizer.decode(e2b))


hello, my name is raz and i like red
hello, my name is raz and i like red


Have BERT classife Alice/Edgar model outputs

In [28]:
seq_len  = 128
batch_size = 32

a function to generate tokens for BERT to classify

In [35]:
# note: this fun creates the batch+labels directly on GPU

# exlcude tokens that BERT will ignore
tokens_to_exclude = [
    eleuTokenizer.encode('\n'),
    eleuTokenizer.encode('\n\n'),
    eleuTokenizer.encode('\r'),
    eleuTokenizer.encode('\t'),
    eleuTokenizer.encode(' '),
    [eleuTokenizer.eos_token_id],
]


def batch_for_bert():
    # batch_size should be even
    half_batch = batch_size // 2

    # initializer batch tensors and labels
    batch2classify = torch.zeros(batch_size, seq_len, dtype=torch.long, device = device)
    labels = torch.zeros(batch_size, dtype=torch.long, device=device)

    # create random starting tokens
    randstarts = torch.randint(eleuTokenizer.vocab_size//2, (half_batch,1)).to(device)

    # generate tokens in batch
    outs_alice = modelAlice.generate(
        randstarts,
        min_length = 4*seq_len,
        max_length = 4*seq_len, # we genrate way too many tokens, to ensure while translating to BERT tokens we get exact seq_len
        do_sample = True,
        early_stopping = False,
        eos_token_id = None,
        pad_token_id = eleuTokenizer.pad_token_id,
        bad_words_ids = tokens_to_exclude,
        repetition_penalty = 1.3
    )


    outs_edgar = modelEdgar.generate(
        randstarts,
        min_length = 4*seq_len,
        max_length = 4*seq_len, # we genrate way too many tokens, to ensure while translating to BERT tokens we get exact seq_len
        do_sample = True,
        early_stopping = False,
        eos_token_id = None,
        pad_token_id = eleuTokenizer.pad_token_id,
        bad_words_ids = tokens_to_exclude,
        repetition_penalty = 1.3
    )

    # fill the batch tensor
    for i in range(half_batch):

        # first 1/2 is alice generated tokens
        outA = eleu2bert(outs_alice[i,1:])
        batch2classify[i,:] = outA[:seq_len]
        labels[i] = 0 # 0 for alice

        # second 1/2 is edghar generated tokens
        outE = eleu2bert(outs_edgar[i,1:])
        batch2classify[half_batch+i,:] = outE[:seq_len]
        labels[half_batch+i] = 1 # 1 for edgar

    del outs_alice, outs_edgar # remove large matrices from memory
    return batch2classify, labels

In [36]:
# test
batch2classify, labels = batch_for_bert()

print(batch2classify.shape)
print(labels.shape)

torch.Size([32, 64])
torch.Size([32])


In [37]:
# BERT loss func
bert_loss_fun = nn.CrossEntropyLoss()

# fwd pass, get model preds, and report loss=acc
logits = bert(batch2classify)
predLabels = torch.argmax(logits, dim=1)
loss = bert_loss_fun(logits, labels).item()

print('\nPredicted labels:\n',predLabels)
print('acutal labels:\n', labels)

print(f'\nLoss: {loss:.4f}')
print(f'\naccuracy: {(predLabels==labels).sum().item()/batch_size}')


Predicted labels:
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='mps:0')
acutal labels:
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1], device='mps:0')

Loss: 1.0742

accuracy: 0.5


Train the EDGAR and ALice models on their texts

In [38]:
# ALICE optmizer
optimizerAlice = torch.optim.AdamW(modelAlice.parameters(), lr=1e-5)
optimizerEdgar = torch.optim.AdamW(modelEdgar.parameters(), lr=1e-5)

num_samples=100

In [41]:

lossAlice = np.zeros(num_samples)
lossEdgar = np.zeros(num_samples)
bertAccuracy = np.zeros(num_samples)


for sampli in range(num_samples):
    # init batch losses to accumulate

    # ALICE fine tuning
    # get a batch of data
    ix = torch.randint(len(aliceTokens)-seq_len, size = (batch_size,))
    X = aliceTokens[ix[:,None] + torch.arange(seq_len)].to(device)

    #fwd pass and get loss
    modelAlice.zero_grad()
    outputs = modelAlice(X, labels=X)

    # backprop and store loss
    outputs.loss.backward()
    optimizerAlice.step()
    lossAlice[sampli] = outputs.loss.item()


    #EDGAR fine tuning
    ix = torch.randint(len(edgarTokens)-seq_len, size = (batch_size,))
    X = edgarTokens[ix[:,None] + torch.arange(seq_len)].to(device)

    #fwd pass and get loss
    modelEdgar.zero_grad()
    outputs = modelEdgar(X, labels=X)

    # backprop and store loss
    outputs.loss.backward()
    optimizerEdgar.step()
    lossEdgar[sampli] = outputs.loss.item()

    if sampli%10==0:
        # have the models generate some text
        batch2classify, labels = batch_for_bert()

        # have BERT classify
        with torch.no_grad(): # turn off grad calculations
            logits = bert(batch2classify) # fwd pass
        predLabels = torch.argmax(logits, dim=1) # get model preds
        bertAccuracy[sampli] = (predLabels == labels).sum().item()/batch_size

        
        print(f'Sample: {sampli}/{num_samples}, losses (Alice/eDgar): {lossAlice[sampli]} / {lossEdgar[sampli]} , BERT classification acc {bertAccuracy[sampli]}')

Sample: 0/100, losses (Alice/eDgar): 3.1275529861450195 / 3.099447250366211 , BERT classification acc 0.5
Sample: 10/100, losses (Alice/eDgar): 2.9386887550354004 / 2.9384987354278564 , BERT classification acc 0.5


Token indices sequence length is longer than the specified maximum sequence length for this model (1533 > 512). Running this sequence through the model will result in indexing errors


Sample: 20/100, losses (Alice/eDgar): 2.8115274906158447 / 2.9203615188598633 , BERT classification acc 0.5
Sample: 30/100, losses (Alice/eDgar): 2.7352123260498047 / 2.839982509613037 , BERT classification acc 0.5
Sample: 40/100, losses (Alice/eDgar): 2.575998544692993 / 2.796680450439453 , BERT classification acc 0.5
Sample: 50/100, losses (Alice/eDgar): 2.683837890625 / 2.6662378311157227 , BERT classification acc 0.5
Sample: 60/100, losses (Alice/eDgar): 2.548097610473633 / 2.705105781555176 , BERT classification acc 0.5
Sample: 70/100, losses (Alice/eDgar): 2.5983614921569824 / 2.63039493560791 , BERT classification acc 0.5
Sample: 80/100, losses (Alice/eDgar): 2.5192208290100098 / 2.673112630844116 , BERT classification acc 0.5
Sample: 90/100, losses (Alice/eDgar): 2.5500779151916504 / 2.6911542415618896 , BERT classification acc 0.5


In [43]:
# If BERT can classify with 90% accuracy which style is being generated by these 2 models, then we can assume the gen models are good
#     The BERT was trained on actual texts from the 2 books
#     The data it classifies does not come from those texts,
#             but from the 2 models that was trained on these books 


In [45]:
# using a 3rd classifier models is good way to evaluate generative models
    # but htis could also produce deceptive results